In [0]:
%sql
SELECT version() AS spark_version, 'Silver notebook started' AS status;

spark_version,status
4.1.0 0000000000000000000000000000000000000000,Silver notebook started


In [0]:
%sql
-- Check Bronze schemas visible to this notebook

SHOW CATALOGS;
SHOW TABLES IN atliq.bronze;

catalog
atliq
dbw_atliq_capstone_jd
samples
system


database,tableName,isTemporary
bronze,customers,false
bronze,order_items,false
bronze,orders,false
bronze,payments,false
bronze,products,false


In [0]:
%sql
-- Schemas inside atliq
SHOW SCHEMAS IN atliq;

databaseName
bronze
ci
default
fabric_gold
gold
information_schema
silver


**Identify the Bronze tables**

In [0]:
%sql
-- List all tables in Bronze

SELECT table_name
FROM atliq.information_schema.tables
WHERE table_schema = 'bronze'
ORDER BY table_name;

table_name
customers
order_items
orders
payments
products


**Inspect all Bronze table schemas**

In [0]:
%sql
-- Inspect Bronze schemas
DESCRIBE TABLE atliq.bronze.customers;
DESCRIBE TABLE atliq.bronze.orders;
DESCRIBE TABLE atliq.bronze.order_items;
DESCRIBE TABLE atliq.bronze.payments;
DESCRIBE TABLE atliq.bronze.products;

col_name,data_type,comment
customer_id,bigint,null
customer_name,string,null
email,string,null
city,string,null
signup_date,date,null
updated_at,timestamp,null
_rescued_data,string,null


col_name,data_type,comment
order_id,bigint,null
customer_id,bigint,null
order_date,date,null
status,string,null
order_amount,"decimal(12,2)",null
created_at,timestamp,null
updated_at,timestamp,null
_rescued_data,string,null


col_name,data_type,comment
order_item_id,bigint,null
order_id,bigint,null
product_id,bigint,null
quantity,bigint,null
item_price,"decimal(10,2)",null
created_at,timestamp,null
_rescued_data,string,null


col_name,data_type,comment
payment_id,bigint,null
order_id,bigint,null
amount,"decimal(12,2)",null
method,string,null
paid_at,timestamp,null
updated_at,timestamp,null
_rescued_data,string,null


col_name,data_type,comment
product_id,bigint,null
product_name,string,null
category,string,null
unit_price,"decimal(10,2)",null
updated_at,timestamp,null
_rescued_data,string,null


**First do a data-quality assessment**

In [0]:
%sql
-- Bronze data profiling and quality assessment

SELECT
  'customers' AS table_name,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
  SUM(CASE WHEN customer_name IS NULL THEN 1 ELSE 0 END) AS customer_name_nulls,
  SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS email_nulls,
  SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS city_nulls,
  SUM(CASE WHEN signup_date IS NULL THEN 1 ELSE 0 END) AS signup_date_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.bronze.customers;

SELECT
  'orders' AS table_name,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS order_date_nulls,
  SUM(CASE WHEN status IS NULL THEN 1 ELSE 0 END) AS status_nulls,
  SUM(CASE WHEN order_amount IS NULL THEN 1 ELSE 0 END) AS order_amount_nulls,
  SUM(CASE WHEN created_at IS NULL THEN 1 ELSE 0 END) AS created_at_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.bronze.orders;

SELECT
  'order_items' AS table_name,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS order_item_id_nulls,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_nulls,
  SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS quantity_nulls,
  SUM(CASE WHEN item_price IS NULL THEN 1 ELSE 0 END) AS item_price_nulls,
  SUM(CASE WHEN created_at IS NULL THEN 1 ELSE 0 END) AS created_at_nulls
FROM atliq.bronze.order_items;

SELECT
  'payments' AS table_name,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN payment_id IS NULL THEN 1 ELSE 0 END) AS payment_id_nulls,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS amount_nulls,
  SUM(CASE WHEN method IS NULL THEN 1 ELSE 0 END) AS method_nulls,
  SUM(CASE WHEN paid_at IS NULL THEN 1 ELSE 0 END) AS paid_at_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.bronze.payments;

SELECT
  'products' AS table_name,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_nulls,
  SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS product_name_nulls,
  SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS category_nulls,
  SUM(CASE WHEN unit_price IS NULL THEN 1 ELSE 0 END) AS unit_price_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.bronze.products;

table_name,total_rows,customer_id_nulls,customer_name_nulls,email_nulls,city_nulls,signup_date_nulls,updated_at_nulls
customers,40,40,0,0,0,0,0


table_name,total_rows,order_id_nulls,customer_id_nulls,order_date_nulls,status_nulls,order_amount_nulls,created_at_nulls,updated_at_nulls
orders,308,308,308,0,0,0,0,0


table_name,total_rows,order_item_id_nulls,order_id_nulls,product_id_nulls,quantity_nulls,item_price_nulls,created_at_nulls
order_items,798,798,798,798,798,0,0


table_name,total_rows,payment_id_nulls,order_id_nulls,amount_nulls,method_nulls,paid_at_nulls,updated_at_nulls
payments,254,254,254,0,0,0,0


table_name,total_rows,product_id_nulls,product_name_nulls,category_nulls,unit_price_nulls,updated_at_nulls
products,25,25,0,0,0,0



**Bronze data profiling and quality assessment**


In [0]:
%sql
-- Inspect Bronze data and rescued records

SELECT 'customers' AS table_name, *
FROM atliq.bronze.customers
LIMIT 5;

SELECT 'orders' AS table_name, *
FROM atliq.bronze.orders
LIMIT 5;

SELECT 'order_items' AS table_name, *
FROM atliq.bronze.order_items
LIMIT 5;

SELECT 'payments' AS table_name, *
FROM atliq.bronze.payments
LIMIT 5;

SELECT 'products' AS table_name, *
FROM atliq.bronze.products
LIMIT 5;

table_name,customer_id,customer_name,email,city,signup_date,updated_at,_rescued_data
customers,null,Rahul Verma,rahul.verma1@example.com,Chennai,2025-11-20,2025-11-20T11:00:00.000Z,"{""customer_id"":1,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/customers/customers.parquet""}"
customers,null,Krishna Patel,krishna.patel2@example.com,Jaipur,2025-10-11,2025-10-11T09:00:00.000Z,"{""customer_id"":2,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/customers/customers.parquet""}"
customers,null,Meera Gupta,meera.gupta3@example.com,Bengaluru,2025-11-16,2025-11-16T09:00:00.000Z,"{""customer_id"":3,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/customers/customers.parquet""}"
customers,null,Reyansh Reddy,reyansh.reddy4@example.com,Surat,2025-03-19,2025-03-19T08:00:00.000Z,"{""customer_id"":4,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/customers/customers.parquet""}"
customers,null,Kavya Reddy,kavya.reddy5@example.com,Jaipur,2024-12-01,2024-12-01T14:00:00.000Z,"{""customer_id"":5,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/customers/customers.parquet""}"


table_name,order_id,customer_id,order_date,status,order_amount,created_at,updated_at,_rescued_data
orders,null,null,2026-01-22,Returned,12095.00,2026-01-22T20:31:00.000Z,2026-01-25T04:31:00.000Z,"{""order_id"":1,""customer_id"":6,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/orders/orders.parquet""}"
orders,null,null,2026-06-05,Placed,4595.00,2026-06-05T15:13:00.000Z,2026-06-05T15:13:00.000Z,"{""order_id"":2,""customer_id"":34,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/orders/orders.parquet""}"
orders,null,null,2026-05-13,Delivered,5445.00,2026-05-13T16:07:00.000Z,2026-05-16T17:07:00.000Z,"{""order_id"":3,""customer_id"":29,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/orders/orders.parquet""}"
orders,null,null,2026-02-28,Delivered,2197.00,2026-02-28T10:57:00.000Z,2026-03-04T11:57:00.000Z,"{""order_id"":4,""customer_id"":4,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/orders/orders.parquet""}"
orders,null,null,2026-05-28,Returned,4396.00,2026-05-28T16:15:00.000Z,2026-06-02T22:15:00.000Z,"{""order_id"":5,""customer_id"":37,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/orders/orders.parquet""}"


table_name,order_item_id,order_id,product_id,quantity,item_price,created_at,_rescued_data
order_items,null,null,null,null,4999.00,2026-01-22T20:31:00.000Z,"{""order_item_id"":1,""order_id"":1,""product_id"":5,""quantity"":2,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/order_items/order_items.parquet""}"
order_items,null,null,null,null,699.00,2026-01-22T20:31:00.000Z,"{""order_item_id"":2,""order_id"":1,""product_id"":22,""quantity"":3,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/order_items/order_items.parquet""}"
order_items,null,null,null,null,799.00,2026-06-05T15:13:00.000Z,"{""order_item_id"":3,""order_id"":2,""product_id"":23,""quantity"":2,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/order_items/order_items.parquet""}"
order_items,null,null,null,null,999.00,2026-06-05T15:13:00.000Z,"{""order_item_id"":4,""order_id"":2,""product_id"":10,""quantity"":3,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/order_items/order_items.parquet""}"
order_items,null,null,null,null,2499.00,2026-05-13T16:07:00.000Z,"{""order_item_id"":5,""order_id"":3,""product_id"":1,""quantity"":1,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/order_items/order_items.parquet""}"


table_name,payment_id,order_id,amount,method,paid_at,updated_at,_rescued_data
payments,null,null,12095.00,Credit Card,2026-01-22T20:40:00.000Z,2026-01-22T20:40:00.000Z,"{""payment_id"":1,""order_id"":1,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/payments/payments.parquet""}"
payments,null,null,4595.00,COD,2026-06-05T15:25:00.000Z,2026-06-05T15:25:00.000Z,"{""payment_id"":2,""order_id"":2,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/payments/payments.parquet""}"
payments,null,null,5445.00,UPI,2026-05-13T16:10:00.000Z,2026-05-13T16:10:00.000Z,"{""payment_id"":3,""order_id"":3,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/payments/payments.parquet""}"
payments,null,null,2197.00,Wallet,2026-02-28T11:02:00.000Z,2026-02-28T11:02:00.000Z,"{""payment_id"":4,""order_id"":4,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/payments/payments.parquet""}"
payments,null,null,4396.00,Net Banking,2026-05-28T16:29:00.000Z,2026-05-28T16:29:00.000Z,"{""payment_id"":5,""order_id"":5,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/payments/payments.parquet""}"


table_name,product_id,product_name,category,unit_price,updated_at,_rescued_data
products,null,Wireless Earbuds,Electronics,2499.00,2025-12-15T10:00:00.000Z,"{""product_id"":1,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/products/products.parquet""}"
products,null,Bluetooth Speaker,Electronics,3299.00,2025-12-15T10:00:00.000Z,"{""product_id"":2,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/products/products.parquet""}"
products,null,Power Bank 20000mAh,Electronics,1799.00,2025-12-15T10:00:00.000Z,"{""product_id"":3,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/products/products.parquet""}"
products,null,USB-C Charger 65W,Electronics,1499.00,2025-12-15T10:00:00.000Z,"{""product_id"":4,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/products/products.parquet""}"
products,null,Smartwatch,Electronics,4999.00,2025-12-15T10:00:00.000Z,"{""product_id"":5,""_file_path"":""abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/raw/products/products.parquet""}"


**Test the rescued IDs**

In [0]:
%sql
-- Test extraction from _rescued_data
SELECT
  customer_id AS bronze_customer_id,
  get_json_object(_rescued_data, '$.customer_id') AS rescued_customer_id,
  customer_name,
  email,
  city,
  signup_date,
  updated_at
FROM atliq.bronze.customers
LIMIT 10;

bronze_customer_id,rescued_customer_id,customer_name,email,city,signup_date,updated_at
null,1,Rahul Verma,rahul.verma1@example.com,Chennai,2025-11-20,2025-11-20T11:00:00.000Z
null,2,Krishna Patel,krishna.patel2@example.com,Jaipur,2025-10-11,2025-10-11T09:00:00.000Z
null,3,Meera Gupta,meera.gupta3@example.com,Bengaluru,2025-11-16,2025-11-16T09:00:00.000Z
null,4,Reyansh Reddy,reyansh.reddy4@example.com,Surat,2025-03-19,2025-03-19T08:00:00.000Z
null,5,Kavya Reddy,kavya.reddy5@example.com,Jaipur,2024-12-01,2024-12-01T14:00:00.000Z
null,6,Krishna Singh,krishna.singh6@example.com,Chennai,2025-02-04,2025-02-04T20:00:00.000Z
null,7,Nikhil Sharma,nikhil.sharma7@example.com,Kolkata,2025-09-12,2025-09-12T13:00:00.000Z
null,8,Ishaan Patel,ishaan.patel8@example.com,Pune,2025-08-14,2025-08-14T09:00:00.000Z
null,9,Aditya Gupta,aditya.gupta9@example.com,Pune,2025-10-14,2025-10-14T13:00:00.000Z
null,10,Neha Nair,neha.nair10@example.com,Ahmedabad,2025-11-10,2025-11-10T16:00:00.000Z


In [0]:
%sql
-- BRONZE → SILVER: CUSTOMERS

CREATE OR REPLACE TABLE atliq.silver.customers
USING DELTA
AS
WITH silver_customers AS (
  SELECT
    CAST(get_json_object(_rescued_data, '$.customer_id') AS BIGINT) AS customer_id,
    TRIM(customer_name) AS customer_name,
    LOWER(TRIM(email)) AS email,
    INITCAP(TRIM(city)) AS city,
    TO_DATE(signup_date) AS signup_date,
    CAST(updated_at AS TIMESTAMP) AS updated_at
  FROM atliq.bronze.customers
), deduped_customers AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY customer_id) AS rn
  FROM silver_customers
  WHERE customer_id IS NOT NULL
)
SELECT
  customer_id,
  customer_name,
  email,
  city,
  signup_date,
  updated_at
FROM deduped_customers
WHERE rn = 1;

SELECT *
FROM atliq.silver.customers
ORDER BY customer_id
LIMIT 10;

num_affected_rows,num_inserted_rows


customer_id,customer_name,email,city,signup_date,updated_at
1,Rahul Verma,rahul.verma1@example.com,Chennai,2025-11-20,2025-11-20T11:00:00.000Z
2,Krishna Patel,krishna.patel2@example.com,Jaipur,2025-10-11,2025-10-11T09:00:00.000Z
3,Meera Gupta,meera.gupta3@example.com,Bengaluru,2025-11-16,2025-11-16T09:00:00.000Z
4,Reyansh Reddy,reyansh.reddy4@example.com,Surat,2025-03-19,2025-03-19T08:00:00.000Z
5,Kavya Reddy,kavya.reddy5@example.com,Jaipur,2024-12-01,2024-12-01T14:00:00.000Z
6,Krishna Singh,krishna.singh6@example.com,Chennai,2025-02-04,2025-02-04T20:00:00.000Z
7,Nikhil Sharma,nikhil.sharma7@example.com,Kolkata,2025-09-12,2025-09-12T13:00:00.000Z
8,Ishaan Patel,ishaan.patel8@example.com,Pune,2025-08-14,2025-08-14T09:00:00.000Z
9,Aditya Gupta,aditya.gupta9@example.com,Pune,2025-10-14,2025-10-14T13:00:00.000Z
10,Neha Nair,neha.nair10@example.com,Ahmedabad,2025-11-10,2025-11-10T16:00:00.000Z


In [0]:
%sql
-- BRONZE → SILVER: ORDERS

CREATE OR REPLACE TABLE atliq.silver.orders
USING DELTA
AS
WITH silver_orders AS (
  SELECT
    CAST(get_json_object(_rescued_data, '$.order_id') AS BIGINT) AS order_id,
    CAST(get_json_object(_rescued_data, '$.customer_id') AS BIGINT) AS customer_id,
    TO_DATE(order_date) AS order_date,
    INITCAP(TRIM(status)) AS status,
    CAST(order_amount AS DECIMAL(12,2)) AS order_amount,
    CAST(created_at AS TIMESTAMP) AS created_at,
    CAST(updated_at AS TIMESTAMP) AS updated_at
  FROM atliq.bronze.orders
), deduped_orders AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_id) AS rn
  FROM silver_orders
  WHERE order_id IS NOT NULL
)
SELECT
  order_id,
  customer_id,
  order_date,
  status,
  order_amount,
  created_at,
  updated_at
FROM deduped_orders
WHERE rn = 1;

SELECT *
FROM atliq.silver.orders
ORDER BY order_id
LIMIT 10;

num_affected_rows,num_inserted_rows


order_id,customer_id,order_date,status,order_amount,created_at,updated_at
1,6,2026-01-22,Returned,12095.00,2026-01-22T20:31:00.000Z,2026-01-25T04:31:00.000Z
2,34,2026-06-05,Placed,4595.00,2026-06-05T15:13:00.000Z,2026-06-05T15:13:00.000Z
3,29,2026-05-13,Delivered,5445.00,2026-05-13T16:07:00.000Z,2026-05-16T17:07:00.000Z
4,4,2026-02-28,Delivered,2197.00,2026-02-28T10:57:00.000Z,2026-03-04T11:57:00.000Z
5,37,2026-05-28,Returned,4396.00,2026-05-28T16:15:00.000Z,2026-06-02T22:15:00.000Z
6,30,2026-07-06,Cancelled,6598.00,2026-07-06T09:43:00.000Z,2026-07-06T09:43:00.000Z
7,22,2026-07-24,Delivered,13994.00,2026-07-24T22:06:00.000Z,2026-07-28T01:06:00.000Z
8,4,2026-06-16,Delivered,3497.00,2026-06-16T17:53:00.000Z,2026-06-18T20:53:00.000Z
9,4,2026-02-12,Shipped,11389.00,2026-02-12T15:00:00.000Z,2026-02-14T15:00:00.000Z
10,19,2026-02-25,Cancelled,1198.00,2026-02-25T09:37:00.000Z,2026-02-25T09:37:00.000Z


In [0]:
%sql
-- BRONZE → SILVER: ORDER ITEMS

CREATE OR REPLACE TABLE atliq.silver.order_items
USING DELTA
AS
WITH silver_order_items AS (
  SELECT
    CAST(get_json_object(_rescued_data, '$.order_item_id') AS BIGINT) AS order_item_id,
    CAST(get_json_object(_rescued_data, '$.order_id') AS BIGINT) AS order_id,
    CAST(get_json_object(_rescued_data, '$.product_id') AS BIGINT) AS product_id,
    CAST(get_json_object(_rescued_data, '$.quantity') AS BIGINT) AS quantity,
    CAST(item_price AS DECIMAL(10,2)) AS item_price,
    CAST(created_at AS TIMESTAMP) AS created_at
  FROM atliq.bronze.order_items
), deduped_order_items AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY order_item_id ORDER BY order_item_id) AS rn
  FROM silver_order_items
  WHERE order_item_id IS NOT NULL
    AND order_id IS NOT NULL
    AND product_id IS NOT NULL
    AND quantity IS NOT NULL
    AND quantity > 0
    AND item_price >= 0
)
SELECT
  order_item_id,
  order_id,
  product_id,
  quantity,
  item_price,
  created_at
FROM deduped_order_items
WHERE rn = 1;

SELECT *
FROM atliq.silver.order_items
ORDER BY order_item_id
LIMIT 10;

num_affected_rows,num_inserted_rows


order_item_id,order_id,product_id,quantity,item_price,created_at
1,1,5,2,4999.00,2026-01-22T20:31:00.000Z
2,1,22,3,699.00,2026-01-22T20:31:00.000Z
3,2,23,2,799.00,2026-06-05T15:13:00.000Z
4,2,10,3,999.00,2026-06-05T15:13:00.000Z
5,3,1,1,2499.00,2026-05-13T16:07:00.000Z
6,3,19,3,799.00,2026-05-13T16:07:00.000Z
7,3,18,1,549.00,2026-05-13T16:07:00.000Z
8,4,9,2,749.00,2026-02-28T10:57:00.000Z
9,4,22,1,699.00,2026-02-28T10:57:00.000Z
10,5,4,2,1499.00,2026-05-28T16:15:00.000Z


In [0]:
%sql
-- BRONZE → SILVER: PAYMENTS

CREATE OR REPLACE TABLE atliq.silver.payments
USING DELTA
AS
WITH silver_payments AS (
  SELECT
    CAST(get_json_object(_rescued_data, '$.payment_id') AS BIGINT) AS payment_id,
    CAST(get_json_object(_rescued_data, '$.order_id') AS BIGINT) AS order_id,
    CAST(amount AS DECIMAL(12,2)) AS amount,
    TRIM(method) AS method,
    CAST(paid_at AS TIMESTAMP) AS paid_at,
    CAST(updated_at AS TIMESTAMP) AS updated_at
  FROM atliq.bronze.payments
), deduped_payments AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY payment_id) AS rn
  FROM silver_payments
  WHERE payment_id IS NOT NULL
    AND order_id IS NOT NULL
    AND amount IS NOT NULL
    AND amount >= 0
    AND method IS NOT NULL
)
SELECT
  payment_id,
  order_id,
  amount,
  method,
  paid_at,
  updated_at
FROM deduped_payments
WHERE rn = 1;

SELECT *
FROM atliq.silver.payments
ORDER BY payment_id
LIMIT 10;

num_affected_rows,num_inserted_rows


payment_id,order_id,amount,method,paid_at,updated_at
1,1,12095.00,Credit Card,2026-01-22T20:40:00.000Z,2026-01-22T20:40:00.000Z
2,2,4595.00,COD,2026-06-05T15:25:00.000Z,2026-06-05T15:25:00.000Z
3,3,5445.00,UPI,2026-05-13T16:10:00.000Z,2026-05-13T16:10:00.000Z
4,4,2197.00,Wallet,2026-02-28T11:02:00.000Z,2026-02-28T11:02:00.000Z
5,5,4396.00,Net Banking,2026-05-28T16:29:00.000Z,2026-05-28T16:29:00.000Z
6,7,13994.00,Wallet,2026-07-24T22:10:00.000Z,2026-07-24T22:10:00.000Z
7,8,3497.00,Net Banking,2026-06-16T18:22:00.000Z,2026-06-16T18:22:00.000Z
8,9,11389.00,Credit Card,2026-02-12T15:07:00.000Z,2026-02-12T15:07:00.000Z
9,11,3748.00,UPI,2026-01-13T18:50:00.000Z,2026-01-13T18:50:00.000Z
10,12,6496.00,UPI,2026-06-22T22:29:00.000Z,2026-06-22T22:29:00.000Z


In [0]:
%sql
-- BRONZE → SILVER: PRODUCTS

CREATE OR REPLACE TABLE atliq.silver.products
USING DELTA
AS
WITH silver_products AS (
  SELECT
    CAST(get_json_object(_rescued_data, '$.product_id') AS BIGINT) AS product_id,
    TRIM(product_name) AS product_name,
    TRIM(category) AS category,
    CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
    CAST(updated_at AS TIMESTAMP) AS updated_at
  FROM atliq.bronze.products
), deduped_products AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY product_id) AS rn
  FROM silver_products
  WHERE product_id IS NOT NULL
    AND product_name IS NOT NULL
    AND category IS NOT NULL
    AND unit_price IS NOT NULL
    AND unit_price >= 0
)
SELECT
  product_id,
  product_name,
  category,
  unit_price,
  updated_at
FROM deduped_products
WHERE rn = 1;

SELECT *
FROM atliq.silver.products
ORDER BY product_id
LIMIT 10;

num_affected_rows,num_inserted_rows


product_id,product_name,category,unit_price,updated_at
1,Wireless Earbuds,Electronics,2499.00,2025-12-15T10:00:00.000Z
2,Bluetooth Speaker,Electronics,3299.00,2025-12-15T10:00:00.000Z
3,Power Bank 20000mAh,Electronics,1799.00,2025-12-15T10:00:00.000Z
4,USB-C Charger 65W,Electronics,1499.00,2025-12-15T10:00:00.000Z
5,Smartwatch,Electronics,4999.00,2025-12-15T10:00:00.000Z
6,Non-stick Frying Pan,Home & Kitchen,899.00,2025-12-15T10:00:00.000Z
7,Electric Kettle,Home & Kitchen,1299.00,2025-12-15T10:00:00.000Z
8,Steel Water Bottle,Home & Kitchen,549.00,2025-12-15T10:00:00.000Z
9,Storage Container Set,Home & Kitchen,749.00,2025-12-15T10:00:00.000Z
10,LED Desk Lamp,Home & Kitchen,999.00,2025-12-15T10:00:00.000Z


In [0]:
%sql
-- SILVER LAYER - FINAL DATA QUALITY VALIDATION

SELECT 'customers' AS table_name, COUNT(*) AS total_rows FROM atliq.silver.customers;
SELECT 'customers' AS table_name, COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_id_count FROM atliq.silver.customers;
SELECT
  'customers' AS table_name,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
  SUM(CASE WHEN customer_name IS NULL THEN 1 ELSE 0 END) AS customer_name_nulls,
  SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS email_nulls,
  SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS city_nulls,
  SUM(CASE WHEN signup_date IS NULL THEN 1 ELSE 0 END) AS signup_date_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.silver.customers;
SELECT * FROM atliq.silver.customers LIMIT 5;

SELECT 'orders' AS table_name, COUNT(*) AS total_rows FROM atliq.silver.orders;
SELECT 'orders' AS table_name, COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_id_count FROM atliq.silver.orders;
SELECT
  'orders' AS table_name,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS order_date_nulls,
  SUM(CASE WHEN status IS NULL THEN 1 ELSE 0 END) AS status_nulls,
  SUM(CASE WHEN order_amount IS NULL THEN 1 ELSE 0 END) AS order_amount_nulls,
  SUM(CASE WHEN created_at IS NULL THEN 1 ELSE 0 END) AS created_at_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.silver.orders;
SELECT * FROM atliq.silver.orders LIMIT 5;

SELECT 'order_items' AS table_name, COUNT(*) AS total_rows FROM atliq.silver.order_items;
SELECT 'order_items' AS table_name, COUNT(*) - COUNT(DISTINCT order_item_id) AS duplicate_order_item_id_count FROM atliq.silver.order_items;
SELECT
  'order_items' AS table_name,
  SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS order_item_id_nulls,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_nulls,
  SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS quantity_nulls,
  SUM(CASE WHEN item_price IS NULL THEN 1 ELSE 0 END) AS item_price_nulls,
  SUM(CASE WHEN created_at IS NULL THEN 1 ELSE 0 END) AS created_at_nulls
FROM atliq.silver.order_items;
SELECT * FROM atliq.silver.order_items LIMIT 5;

SELECT 'payments' AS table_name, COUNT(*) AS total_rows FROM atliq.silver.payments;
SELECT 'payments' AS table_name, COUNT(*) - COUNT(DISTINCT payment_id) AS duplicate_payment_id_count FROM atliq.silver.payments;
SELECT
  'payments' AS table_name,
  SUM(CASE WHEN payment_id IS NULL THEN 1 ELSE 0 END) AS payment_id_nulls,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS amount_nulls,
  SUM(CASE WHEN method IS NULL THEN 1 ELSE 0 END) AS method_nulls,
  SUM(CASE WHEN paid_at IS NULL THEN 1 ELSE 0 END) AS paid_at_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.silver.payments;
SELECT * FROM atliq.silver.payments LIMIT 5;

SELECT 'products' AS table_name, COUNT(*) AS total_rows FROM atliq.silver.products;
SELECT 'products' AS table_name, COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_id_count FROM atliq.silver.products;
SELECT
  'products' AS table_name,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_nulls,
  SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS product_name_nulls,
  SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS category_nulls,
  SUM(CASE WHEN unit_price IS NULL THEN 1 ELSE 0 END) AS unit_price_nulls,
  SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulls
FROM atliq.silver.products;
SELECT * FROM atliq.silver.products LIMIT 5;

table_name,total_rows
customers,40


table_name,duplicate_customer_id_count
customers,0


table_name,customer_id_nulls,customer_name_nulls,email_nulls,city_nulls,signup_date_nulls,updated_at_nulls
customers,0,0,0,0,0,0


customer_id,customer_name,email,city,signup_date,updated_at
1,Rahul Verma,rahul.verma1@example.com,Chennai,2025-11-20,2025-11-20T11:00:00.000Z
2,Krishna Patel,krishna.patel2@example.com,Jaipur,2025-10-11,2025-10-11T09:00:00.000Z
3,Meera Gupta,meera.gupta3@example.com,Bengaluru,2025-11-16,2025-11-16T09:00:00.000Z
4,Reyansh Reddy,reyansh.reddy4@example.com,Surat,2025-03-19,2025-03-19T08:00:00.000Z
5,Kavya Reddy,kavya.reddy5@example.com,Jaipur,2024-12-01,2024-12-01T14:00:00.000Z


table_name,total_rows
orders,308


...

payment_id,order_id,amount,method,paid_at,updated_at
1,1,12095.00,Credit Card,2026-01-22T20:40:00.000Z,2026-01-22T20:40:00.000Z
2,2,4595.00,COD,2026-06-05T15:25:00.000Z,2026-06-05T15:25:00.000Z
3,3,5445.00,UPI,2026-05-13T16:10:00.000Z,2026-05-13T16:10:00.000Z
4,4,2197.00,Wallet,2026-02-28T11:02:00.000Z,2026-02-28T11:02:00.000Z
5,5,4396.00,Net Banking,2026-05-28T16:29:00.000Z,2026-05-28T16:29:00.000Z


table_name,total_rows
products,25


table_name,duplicate_product_id_count
products,0


table_name,product_id_nulls,product_name_nulls,category_nulls,unit_price_nulls,updated_at_nulls
products,0,0,0,0,0


product_id,product_name,category,unit_price,updated_at
1,Wireless Earbuds,Electronics,2499.00,2025-12-15T10:00:00.000Z
2,Bluetooth Speaker,Electronics,3299.00,2025-12-15T10:00:00.000Z
3,Power Bank 20000mAh,Electronics,1799.00,2025-12-15T10:00:00.000Z
4,USB-C Charger 65W,Electronics,1499.00,2025-12-15T10:00:00.000Z
5,Smartwatch,Electronics,4999.00,2025-12-15T10:00:00.000Z


**Marketing Spend**

In [0]:
%sql
CREATE OR REPLACE TABLE atliq.silver.marketing_spend
USING DELTA
AS
SELECT
    CAST(spend_date AS DATE) AS spend_date,
    TRIM(channel) AS channel,
    TRIM(campaign) AS campaign,
    CAST(spend_amount AS DECIMAL(18,2)) AS spend_amount,
    CAST(clicks AS INT) AS clicks
FROM read_files(
    'abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/bronze/marketing_spend.csv',
    format => 'csv',
    header => true
);

num_affected_rows,num_inserted_rows


**Supplier Price List**

In [0]:
%sql
CREATE OR REPLACE TABLE atliq.silver.supplier_price_list
USING DELTA
AS
SELECT
    CAST(product_id AS INT) AS product_id,
    TRIM(product_name) AS product_name,
    TRIM(supplier_name) AS supplier_name,
    CAST(supplier_cost AS DECIMAL(18,2)) AS supplier_cost,
    CAST(effective_date AS DATE) AS effective_date
FROM read_files(
    'abfss://lakehouse@atliqcapstonelakejd.dfs.core.windows.net/bronze/supplier_price_list.csv',
    format => 'csv',
    header => true
);

num_affected_rows,num_inserted_rows
